In [ ]:
import mailbox
import json
import os
import uuid
from email.utils import parsedate_to_datetime

In [ ]:
data_dir = "../data"
output_dir = os.path.join(data_dir, "1_parser")
attachments_dir = os.path.join(output_dir, "attachments")
os.makedirs(attachments_dir, exist_ok=True)

mbox_path = os.path.join(data_dir, "mails.mbox")
# mbox_path = os.path.join(data_dir, "Mails_slim.mbox")

In [ ]:
# Load mbox
mbox = mailbox.mbox(mbox_path)

In [ ]:
def get_body(msg):
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            if content_type == "text/plain":
                return part.get_payload(decode=True).decode(errors='ignore')
    else:
        return msg.get_payload(decode=True).decode(errors='ignore')
    return ""


In [ ]:
def get_attachments(msg, email_id, message_id):
    attachments = []    
    attachment_count = 0
    if msg.is_multipart():
        for part in msg.walk():
            content_disposition = part.get("Content-Disposition", "")
            if "attachment" in content_disposition:
                filename = part.get_filename()
                extension = ""  # Initialize extension variable
                if filename:
                    _, extension = os.path.splitext(filename)
                    # Remove non-alphanumeric characters from extension
                    extension = ''.join(c for c in extension if c.isalnum())
                    if extension:
                        extension = '.' + extension
                attachment_path = f"{email_id}_{message_id}_{attachment_count}{extension if extension else '.bin'}"
                full_path = os.path.join(attachments_dir, attachment_path)
                
                # Try different decoding methods for binary files
                payload = None
                try:
                    # For binary files, handle encoding properly
                    content_encoding = part.get('Content-Transfer-Encoding', '')
                    if content_encoding.lower() == 'base64':
                        import base64
                        raw_payload = part.get_payload()
                        # Remove whitespace and newlines from base64 data
                        if isinstance(raw_payload, str):
                            clean_payload = ''.join(raw_payload.split())
                            payload = base64.b64decode(clean_payload)
                        else:
                            payload = raw_payload
                    else:
                        # Try with decode=True first
                        payload = part.get_payload(decode=True)
                        
                except Exception as e:
                    try:
                        # Fallback: try manual base64 decode
                        raw_payload = part.get_payload()
                        if isinstance(raw_payload, str):
                            import base64
                            clean_payload = ''.join(raw_payload.split())
                            payload = base64.b64decode(clean_payload)
                        else:
                            payload = raw_payload
                    except Exception as e2:
                        continue
                
                if not payload:
                    continue
                    
                try:
                    os.makedirs(os.path.dirname(full_path), exist_ok=True)
                    with open(full_path, "wb") as f:
                        f.write(payload)
                except Exception as e:
                    continue
                    
                attachments.append({
                    "id":attachment_count,
                    "path": attachment_path,
                    "original_filename": filename
                })
                attachment_count += 1
    return attachments

In [ ]:
for idx, msg in enumerate(mbox):    
    email_id = str(uuid.uuid4())
    start_date = parsedate_to_datetime(msg["date"])
    start_date_str = start_date.strftime("%Y-%m-%d %H:%M")
    
    sender = msg["from"]
    receiver = msg["to"]

    # Basic structure
    email_json = {
        "Id": email_id,
        "channel": "Mail",
        "StartDate": start_date_str,
        "EndDate": start_date_str,
        "Sender": sender,
        "Receiver": receiver,
        "Messages": []
    }
    
    message_id = str(1)
    message = {
        "Id": message_id,
        "Text": get_body(msg),
        "Subject": msg["subject"],
        "Timestamp": start_date_str,
    }
    
    message["Attachments"] = get_attachments(msg, email_id, message_id)

    email_json["Messages"].append(message)
    # Save results to JSON file
    output_file = os.path.join(output_dir, f"{email_id}.json")
    with open(output_file, "w") as f:
        json.dump(email_json, f, indent=2)
    print(f"Processed email {idx + 1}/{len(mbox)}: {email_id}")